[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C06_Interpretability_Course/01_linear_probes/01_linear_probes.ipynb)

# 01 · Linear Probes：表示即向量

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy + matplotlib，无任何其他依赖，全部 cell 秒级跑完。

**本 notebook 你将完成：**

1. 构造 ground truth 已知的**合成激活空间**（$d=64$：概念方向 + 干扰方向 + 各向异性噪声）；
2. **从零实现 logistic regression probe**（手写梯度下降 + $L_2$ 正则）并在 held-out 集上评估；
3. 实现 **mean-difference probe**，对比两种 probe 的准确率与方向忠实度（与真方向的余弦相似度）；
4. 复现 control task 实验：小样本 + 弱正则下，probe 用**随机标签**也能"拿高分"（capacity 作弊）；
5. 可视化激活在概念方向上的**投影直方图**；
6. 完成 4 道 ✏️ 练习（mean-diff 方向 / accuracy & AUC / selectivity / projection ablation）。

参考文献：Alain & Bengio 2016 (arXiv:1610.01644) · Hewitt & Liang 2019 (arXiv:1909.03368) · Marks & Tegmark 2023 (arXiv:2310.06824) · Park et al. 2023 (arXiv:2311.03658)

## 1 · 合成激活数据：把概念埋进 64 维空间

我们不加载真实模型，而是构造一个**全部真相已知**的合成"激活空间"——这是校准方法论直觉的正确顺序：先在沙盒里知道答案，再上真模型。

生成模型（模拟某层 residual stream 的激活 $h \in \mathbb{R}^{64}$）：

$$ h = \underbrace{(2y-1)\,\tfrac{\alpha}{2}\, v_{\text{true}}}_{\text{概念信号}} \;+\; \underbrace{s\, v_{\text{distract}}}_{\text{干扰方向}} \;+\; \underbrace{\varepsilon}_{\text{各向异性噪声}} $$

- $v_{\text{true}}$：埋入的**真概念方向**（单位向量），标签 $y\in\{0,1\}$ 只通过它进入激活；
- $v_{\text{distract}}$：与 $v_{\text{true}}$ 正交、方差很大（$s \sim \mathcal{N}(0, 2.5^2)$）但**与标签无关**的干扰特征——模拟真实激活里"声音很大但与你关心的概念无关"的方向；
- $\varepsilon \sim \mathcal{N}(0, \Sigma)$，$\Sigma = Q\,\mathrm{diag}(\lambda)\,Q^\top$，特征值 $\lambda_i = 3\cdot 0.9^i + 0.05$ 衰减——**各向异性**，模拟 residual stream 中少数方向方差极大的现象。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
D, N, ALPHA = 64, 600, 2.0

# 真概念方向（单位向量）
v_true = rng.normal(size=D)
v_true /= np.linalg.norm(v_true)

# 干扰方向：对 v_true 正交化，保证与概念信号"几何上无关"
v_distract = rng.normal(size=D)
v_distract -= (v_distract @ v_true) * v_true
v_distract /= np.linalg.norm(v_distract)

# 各向异性噪声协方差：随机正交基 Q + 衰减特征值谱
Q, _ = np.linalg.qr(rng.normal(size=(D, D)))
eigs = 3.0 * 0.9 ** np.arange(D) + 0.05

y = (rng.random(N) < 0.5).astype(int)          # 标签
signs = 2 * y - 1                              # {0,1} -> {-1,+1}
noise = (rng.normal(size=(N, D)) * np.sqrt(eigs)) @ Q.T
distract = rng.normal(0, 2.5, size=N)[:, None] * v_distract
X = 0.5 * ALPHA * signs[:, None] * v_true + distract + noise

n_train = 480
X_tr, y_tr = X[:n_train], y[:n_train]
X_te, y_te = X[n_train:], y[n_train:]

# 理论信噪比：信号沿 v_true 分离 ALPHA，噪声沿 v_true 的方差 = v^T Σ v
var_along_v = float(((Q.T @ v_true) ** 2 * eigs).sum())
print(f"X: {X.shape}, train {X_tr.shape[0]} / test {X_te.shape[0]}, 正类占比 {y.mean():.2f}")
print(f"噪声沿 v_true 的方差 = {var_along_v:.3f}  ->  d' = {ALPHA/np.sqrt(var_along_v):.2f}")
print(f"v_distract ⟂ v_true: 内积 = {v_distract @ v_true:.1e}")

## 2 · 从零实现 logistic regression probe

probe 模型：$p(y{=}1\mid h) = \sigma(w^\top h + b)$。训练目标 = 交叉熵 + $L_2$：

$$ \mathcal{L}(w,b) = -\frac{1}{n}\sum_i \big[ y_i \log p_i + (1-y_i)\log(1-p_i) \big] + \lambda \lVert w \rVert_2^2 $$

梯度有封闭形式（推导见讲解页 §3.2）：记 $p = \sigma(Hw + b)$，

$$ \nabla_w \mathcal{L} = \tfrac{1}{n} H^\top (p - y) + 2\lambda w, \qquad \partial_b \mathcal{L} = \tfrac{1}{n}\textstyle\sum_i (p_i - y_i) $$

损失是凸的（$\lambda>0$ 时强凸），普通梯度下降必收敛到全局最优——probe 没有调参玄学，适合当测量仪器。注意 $\lambda$ 的双重身份：既保证可分数据下解有限，又控制 probe 的"记忆容量"（第 4 节）。

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))   # clip 防溢出

def train_logistic_probe(X, y, l2=1e-2, lr=0.5, n_steps=2000):
    """手写梯度下降训练 logistic probe，返回 (w, b, losses)。"""
    n, d = X.shape
    w, b = np.zeros(d), 0.0
    losses = []
    for _ in range(n_steps):
        p = sigmoid(X @ w + b)
        eps = 1e-12
        losses.append(float(-np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
                            + l2 * np.sum(w ** 2)))
        gw = X.T @ (p - y) / n + 2 * l2 * w
        gb = float(np.mean(p - y))
        w -= lr * gw
        b -= lr * gb
    return w, b, losses

def probe_acc(X, y, w, b):
    return float(((sigmoid(X @ w + b) > 0.5).astype(int) == y).mean())

w_lr, b_lr, losses = train_logistic_probe(X_tr, y_tr)
print(f"trained probe: train acc = {probe_acc(X_tr, y_tr, w_lr, b_lr):.3f}, "
      f"test acc = {probe_acc(X_te, y_te, w_lr, b_lr):.3f}")
assert losses[-1] < losses[0]

plt.figure(figsize=(6, 3.2))
plt.plot(losses)
plt.xlabel("GD step"); plt.ylabel("loss")
plt.title("logistic probe training loss (convex -> monotone)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 3 · mean-difference probe：零训练的对照

最简单的 probe 不用训练——直接拿两类激活的均值差作方向：

$$ w_{\text{md}} = \mu^+ - \mu^-, \qquad b_{\text{md}} = -\,w_{\text{md}}^\top \frac{\mu^+ + \mu^-}{2} $$

理论关系：同协方差高斯下 Bayes 最优方向是 LDA 方向 $\Sigma^{-1}(\mu^+ - \mu^-)$；logistic probe 渐近逼近它，所以**各向异性噪声下分类更准**；但代价是 $w$ 混入"抵消噪声协方差"的成分，**方向不再纯指概念**。预期结果（[Marks & Tegmark 2023] 的合成版）：

| | 准确率 | 与 $v_{\text{true}}$ 的余弦 |
|---|---|---|
| trained probe | 更高 | 更低（方向被 $\Sigma^{-1}$ 拉偏） |
| mean-diff probe | 较低 | **更高（方向更忠实）** |

要分类用 trained probe；要拿方向去做干预/几何分析，mean-diff 往往更可靠。

In [ ]:
mu1, mu0 = X_tr[y_tr == 1].mean(axis=0), X_tr[y_tr == 0].mean(axis=0)
w_md = mu1 - mu0
b_md = float(-w_md @ ((mu1 + mu0) / 2))

def cos(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

rows = [
    ("trained (logistic)", probe_acc(X_te, y_te, w_lr, b_lr), cos(w_lr, v_true)),
    ("mean-difference   ", probe_acc(X_te, y_te, w_md, b_md), cos(w_md, v_true)),
]
print(f"{'probe':22s} {'test acc':>9s} {'cos(w, v_true)':>15s}")
for name, acc, c in rows:
    print(f"{name:22s} {acc:9.3f} {c:15.3f}")

# 准确率：trained 赢；方向忠实度：mean-diff 赢
assert rows[0][1] > rows[1][1] and rows[1][2] > rows[0][2]
print("\n=> trained probe 分类更准，mean-diff 方向更忠实 —— 两个不同的优化目标")

## 4 · control task：高准确率可以是 capacity 买来的

[Hewitt & Liang 2019] 的核心实验：把标签换成**随机标签**（与激活内容完全无关），如果 probe 还能拿高分，说明它在靠**容量记忆**，而不是在读激活里的结构。

几何直觉（Cover 定理）：$d$ 维空间中 $n$ 个一般位置的点配随机标签，当 $n \le d+1$ 时**必然**线性可分，$n \le 2d$ 时大概率可分。我们的 $d=64$，取 $n=80$ 个训练样本 + 弱正则（$\lambda=10^{-4}$）——看 probe 怎么"作弊"。

$$ \mathrm{selectivity} = \mathrm{acc}_{\text{task}} - \mathrm{acc}_{\text{control}} $$

**报告 probe 结果必须同时报告 control 基线**，且评估一律用 held-out 集。

In [ ]:
rng_ctrl = np.random.default_rng(0)
n_small = 80
X_small = X_tr[:n_small]
y_rand = rng_ctrl.integers(0, 2, n_small)          # 与激活完全无关的随机标签

w_c, b_c, _ = train_logistic_probe(X_small, y_rand, l2=1e-4, n_steps=4000)
acc_ctrl_train = probe_acc(X_small, y_rand, w_c, b_c)

y_rand_te = rng_ctrl.integers(0, 2, len(X_te))     # held-out 也配随机标签
acc_ctrl_test = probe_acc(X_te, y_rand_te, w_c, b_c)

print(f"control task (n={n_small}, d={D}, l2=1e-4):")
print(f"  train acc = {acc_ctrl_train:.3f}   <- 随机标签也被'学会'了（纯记忆）")
print(f"  test  acc = {acc_ctrl_test:.3f}   <- held-out 立刻打回随机水平")
assert acc_ctrl_train > 0.95 and abs(acc_ctrl_test - 0.5) < 0.15

print("\n教训：n ≲ d 是红区；probe 准确率必须配 control 基线 + held-out 评估才可信")

## 5 · 可视化：激活在概念方向上的投影

LRH 的"measurement"含义可以直接画出来：把每个激活投影到 $v_{\text{true}}$（标量 $h^\top v_{\text{true}}$），两类应分成两座小山；投影到**干扰方向**则两类完全重叠。

**本节小结**（带着这些结论进练习）：

- probe = 激活空间上的线性分类器；trained probe 准确率高，mean-diff 方向忠实——按用途选择；
- probe 准确率只证明**线性可解码**，不证明模型**使用**（要证使用得做干预，练习 4 的 projection ablation 是最简形式）；
- 高准确率可能是容量作弊：报告 selectivity（练习 3），评估永远 held-out。

In [ ]:
proj_true = X_te @ v_true        # 概念方向上的投影
proj_dist = X_te @ v_distract    # 干扰方向上的投影

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, proj, name in [(axes[0], proj_true, "concept direction $v_{true}$"),
                       (axes[1], proj_dist, "distractor direction")]:
    ax.hist(proj[y_te == 0], bins=24, alpha=0.6, label="y=0", color="#d62728")
    ax.hist(proj[y_te == 1], bins=24, alpha=0.6, label="y=1", color="#1f77b4")
    ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.set_xlabel("projection"); ax.set_title(name); ax.legend()
axes[0].set_ylabel("count")
plt.tight_layout(); plt.show()

print("左图：概念方向上两类分离（这就是 probe 能工作的全部原因）")
print("右图：干扰方向方差更大，但两类完全重叠 —— 响度 != 相关性")

---
## ✏️ 练习 1：实现 mean-difference probe 方向

实现 `mean_diff_direction(X, y)`：返回**单位化**的均值差方向 $\hat w = \dfrac{\mu^+ - \mu^-}{\lVert \mu^+ - \mu^- \rVert}$（$\mu^+$ 为 `y==1` 类均值）。

**提示**：布尔索引 `X[y == 1].mean(axis=0)` 一行得到类均值；别忘了除以范数；方向有正负——必须是 $\mu^+ - \mu^-$ 而不是反过来。5 行以内。

In [ ]:
def mean_diff_direction(X, y):
    # TODO: 计算 mu_plus(y==1 类均值) - mu_minus(y==0 类均值)，单位化后返回
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
v_md = mean_diff_direction(X, y)
assert v_md.shape == (D,)
assert abs(np.linalg.norm(v_md) - 1.0) < 1e-8          # 必须单位化
assert cos(v_md, v_true) > 0.9                         # 与埋入的真方向高度对齐（且符号正确）
toy = mean_diff_direction(np.array([[0., 0.], [0., 0.], [2., 2.], [2., 2.]]),
                          np.array([0, 0, 1, 1]))
assert np.allclose(toy, [2 ** -0.5, 2 ** -0.5])        # 玩具样例：方向 = (1,1)/sqrt(2)
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 probe 的 accuracy 与 AUC

实现 `probe_metrics(scores, y, threshold=0.0)`，输入 probe 的原始分数 $s_i = w^\top h_i + b$ 与标签，返回 `(acc, auc)`：

- **acc**：`scores > threshold` 作为预测，与 `y` 的符合率；
- **auc**：Mann–Whitney 形式 $\mathrm{AUC} = \Pr(s^+ > s^-)$，用**秩统计量**免去逐对比较：把分数从小到大排秩（1 起），则 $\mathrm{AUC} = \dfrac{\sum_{i: y_i=1} r_i - n^+(n^++1)/2}{n^+ n^-}$。

**提示**：`order = np.argsort(scores)` 后 `ranks[order] = np.arange(1, n+1)` 两行得到秩；分数连续无并列，不用处理 ties。约 10 行。

In [ ]:
def probe_metrics(scores, y, threshold=0.0):
    # TODO:
    # 1) acc = (scores > threshold) 与 y 的符合率
    # 2) ranks: argsort 后填 1..n
    # 3) auc = (正类秩和 - n_pos*(n_pos+1)/2) / (n_pos*n_neg)
    # 返回 (acc, auc)，都是 float
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
acc, auc = probe_metrics(np.array([-2., -1., 1., 2.]), np.array([0, 0, 1, 1]))
assert acc == 1.0 and auc == 1.0                       # 完美分离
acc, auc = probe_metrics(np.array([-2., -1., 1., 2.]), np.array([1, 1, 0, 0]))
assert acc == 0.0 and auc == 0.0                       # 完美反向
acc, auc = probe_metrics(np.array([1., 2., 3., 4.]), np.array([0, 1, 0, 1]))
assert abs(auc - 0.75) < 1e-9                          # 4 对正负样本中 3 对排序正确
assert abs(acc - 0.5) < 1e-9                           # threshold=0 全预测正类 -> 0.5
acc_lr, auc_lr = probe_metrics(X_te @ w_lr + b_lr, y_te)
assert auc_lr > 0.95                                   # 第 2 节的 probe 在 held-out 上 AUC 很高
print(f"trained probe held-out: acc={acc_lr:.3f}, AUC={auc_lr:.3f}")
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 control task selectivity

实现 `selectivity_experiment(X, y, n_train, l2, n_steps=3000)`：

1. 取前 `n_train` 个样本作训练集；
2. control 标签 = `np.random.default_rng(0).permutation(y_sub)`（打乱后与激活无关，但保持类别比例）；
3. 用同样的超参分别在真标签与 control 标签上训练 logistic probe（直接调上面的 `train_logistic_probe`）；
4. 返回 `(acc_task, acc_control, selectivity)`——这里两个 acc 都取**训练集**准确率（control task 诊断的正是"训练集上的记忆能力"），selectivity = 差值。

**提示**：约 8 行。预期现象——大样本 + 正常正则（`n_train=400, l2=1e-2`）selectivity 高；小样本 + 弱正则（`n_train=80, l2=1e-4`）probe 把随机标签也背下来，selectivity 崩到 ≈ 0。

In [ ]:
def selectivity_experiment(X, y, n_train, l2, n_steps=3000):
    # TODO:
    # 1) X_sub, y_sub = 前 n_train 个样本
    # 2) y_ctrl = np.random.default_rng(0).permutation(y_sub)
    # 3) 分别训练 probe（train_logistic_probe，超参相同），算各自的训练集 acc
    # 4) return (acc_task, acc_control, acc_task - acc_control)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
acc_t1, acc_c1, sel1 = selectivity_experiment(X, y, n_train=400, l2=1e-2)
acc_t2, acc_c2, sel2 = selectivity_experiment(X, y, n_train=80, l2=1e-4, n_steps=4000)
assert -1.0 <= sel1 <= 1.0 and -1.0 <= sel2 <= 1.0
assert acc_c1 < 0.75                  # 大样本+正则：随机标签学不动
assert sel1 > 0.25                    # 真概念可分 -> selectivity 高
assert acc_c2 > 0.9                   # 小样本+弱正则：随机标签全背下来（capacity 作弊）
assert sel2 < 0.1 and sel2 < sel1     # selectivity 崩塌，暴露作弊
print(f"n=400, l2=1e-2 : task={acc_t1:.3f} control={acc_c1:.3f} selectivity={sel1:.3f}")
print(f"n= 80, l2=1e-4 : task={acc_t2:.3f} control={acc_c2:.3f} selectivity={sel2:.3f}")
print("✅ 练习 3 通过")

## ✏️ 练习 4：projection ablation —— 从相关到（最简的）因果

实现 `project_out(X, v)`：把每个激活在方向 $v$ 上的分量清零，

$$ h' = (I - \hat v \hat v^\top)\, h, \qquad \hat v = v / \lVert v \rVert $$

然后验证讲解页 §5 的论断：**消除真概念方向后，重新训练的 probe 也只能掉回随机水平**（信息真的没了，不是 probe 没找到）；而消除一个随机方向几乎不伤 probe。

**提示**：核心一行——`X - np.outer(X @ v_hat, v_hat)`；先单位化 `v`；不要原地修改输入。边界检查：ablation 后 `X_abl @ v` 应全为 0。5 行以内。

In [ ]:
def project_out(X, v):
    # TODO: 单位化 v -> v_hat；返回 X - (X @ v_hat) 外积 v_hat（不要修改原 X）
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
X_tr_abl, X_te_abl = project_out(X_tr, v_true), project_out(X_te, v_true)
assert X_tr_abl.shape == X_tr.shape
assert np.allclose(X_tr_abl @ v_true, 0, atol=1e-8)    # v_true 分量被精确清零
assert not np.allclose(X_tr, X_tr_abl)                 # 没有原地修改/空操作

w_a, b_a, _ = train_logistic_probe(X_tr_abl, y_tr, l2=1e-2)
acc_abl = probe_acc(X_te_abl, y_te, w_a, b_a)
assert acc_abl < 0.65                                  # 概念方向被消除 -> 任何 probe 掉回随机

v_rand = np.random.default_rng(1).normal(size=D)       # 种子 1：rng(0) 的第一笔抽样就是 v_true 本身
w_r, b_r, _ = train_logistic_probe(project_out(X_tr, v_rand), y_tr, l2=1e-2)
acc_rand = probe_acc(project_out(X_te, v_rand), y_te, w_r, b_r)
assert acc_rand > 0.85                                 # 消除随机方向几乎无损

print(f"ablate v_true : test acc = {acc_abl:.3f}  (随机水平)")
print(f"ablate random : test acc = {acc_rand:.3f}  (几乎无损)")
print("✅ 练习 4 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def mean_diff_direction(X, y):
    w = X[y == 1].mean(axis=0) - X[y == 0].mean(axis=0)
    return w / np.linalg.norm(w)

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def probe_metrics(scores, y, threshold=0.0):
    scores, y = np.asarray(scores, dtype=float), np.asarray(y)
    acc = float(((scores > threshold).astype(int) == y).mean())
    n = len(scores)
    order = np.argsort(scores)
    ranks = np.empty(n)
    ranks[order] = np.arange(1, n + 1)
    n_pos = int(y.sum()); n_neg = n - n_pos
    auc = float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))
    return acc, auc

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def selectivity_experiment(X, y, n_train, l2, n_steps=3000):
    X_sub, y_sub = X[:n_train], y[:n_train]
    y_ctrl = np.random.default_rng(0).permutation(y_sub)
    w1, b1, _ = train_logistic_probe(X_sub, y_sub, l2=l2, n_steps=n_steps)
    w2, b2, _ = train_logistic_probe(X_sub, y_ctrl, l2=l2, n_steps=n_steps)
    acc_task = probe_acc(X_sub, y_sub, w1, b1)
    acc_control = probe_acc(X_sub, y_ctrl, w2, b2)
    return acc_task, acc_control, acc_task - acc_control

In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def project_out(X, v):
    v_hat = v / np.linalg.norm(v)
    return X - np.outer(X @ v_hat, v_hat)

---
## 🎯 真实数据胶囊题：真实 GPT-2 embedding 上的线性 probe（数字 vs 字母）

linear probe 检验“某个概念是否线性可读”。下载真实 GPT-2 token embedding，训练一个 probe 判断“这个 token 是数字还是字母”，看概念是否线性编码在 embedding 里。

> 本模块新增的**真实数据**练习：用**真实 GPT-2 权重/embedding**把本章的可解释性技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, struct, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.interp_data"); os.makedirs(CACHE,exist_ok=True)
ST="https://huggingface.co/openai-community/gpt2/resolve/main/model.safetensors"
def _rng(s,e):
    req=urllib.request.Request(ST, headers={"Range":f"bytes={s}-{e}"})
    return urllib.request.urlopen(req,timeout=60).read()
def gpt2_emb_block(n=6000):
    cache=os.path.join(CACHE,f"wte_{n}.npy")
    if os.path.exists(cache): return np.load(cache)
    hlen=struct.unpack("<Q", _rng(0,7))[0]; hdr=json.loads(_rng(8,8+hlen-1))
    info=hdr["wte.weight"]; base=8+hlen; s0=info["data_offsets"][0]; d=info["shape"][1]
    raw=_rng(base+s0, base+s0+n*d*4-1)
    E=np.frombuffer(raw,dtype=np.float32).reshape(n,d).copy()
    np.save(cache,E); return E
def gpt2_vocab():
    p=os.path.join(CACHE,"vocab.json")
    if not os.path.exists(p): urllib.request.urlretrieve("https://huggingface.co/openai-community/gpt2/resolve/main/vocab.json",p)
    return json.load(open(p))
def digit_letter_dataset(lim=6000):
    "返回 (X[token嵌入], y[1=数字 0=字母], E, ids_digit, ids_alpha)"
    v=gpt2_vocab(); E=gpt2_emb_block(lim)
    dig=[i for t,i in v.items() if i<lim and t.isdigit()]
    alpha=[i for t,i in v.items() if i<lim and t.isalpha() and t.isascii()]
    rng=np.random.default_rng(0); alpha=list(rng.permutation(alpha)[:len(dig)])
    ids=dig+alpha; y=np.array([1]*len(dig)+[0]*len(alpha))
    return E[ids], y, E, dig, alpha
def shakespeare():
    p=os.path.join(CACHE,"shake.txt")
    if not os.path.exists(p): urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",p)
    return open(p).read()

X, y, E, dig, alpha = digit_letter_dataset()
print(f"真实 GPT-2 embedding 数据: X{X.shape}  数字{int(y.sum())} 字母{int((1-y).sum())}")

**练习**：实现 `train_probe(X, y, steps, lr)`（逻辑回归，标准化输入），返回 `(w, b, mu, sd)`；和 `probe_acc(X, y, params)`。在留出集上准确率应明显 > 0.5（概念线性可读）。

In [ ]:
def train_probe(X, y, steps=500, lr=0.5):
    # TODO: 标准化 X(存 mu,sd)；逻辑回归 GD；返回 (w,b,mu,sd)
    raise NotImplementedError
def probe_acc(X, y, params):
    # TODO: 用 params 标准化并预测，返回准确率
    raise NotImplementedError


In [ ]:
# 自测
rng=np.random.default_rng(1); idx=rng.permutation(len(y)); cut=int(0.7*len(idx))
tr,te=idx[:cut],idx[cut:]
params=train_probe(X[tr],y[tr])
acc=probe_acc(X[te],y[te],params)
assert acc>0.8, f"数字/字母概念应线性可读(>0.8), 得到{acc:.2f}"
print(f"probe 在真实 GPT-2 embedding 上 test acc={acc:.2f} ✓ (数字方向线性可读)")


### 📖 参考答案

In [ ]:
def train_probe(X, y, steps=500, lr=0.5):
    mu,sd=X.mean(0),X.std(0)+1e-8; Xs=(X-mu)/sd
    w=np.zeros(X.shape[1]); b=0.0
    for _ in range(steps):
        p=1/(1+np.exp(-(Xs@w+b))); g=p-y
        w-=lr*Xs.T@g/len(y); b-=lr*g.mean()
    return w,b,mu,sd
def probe_acc(X, y, params):
    w,b,mu,sd=params; p=1/(1+np.exp(-(((X-mu)/sd)@w+b)))
    return float(((p>0.5)==y).mean())
print("✓ probe 高准确率=概念线性可读，但要配 control task 排除'容量买来的'假象")